<a href="https://colab.research.google.com/github/rizwinmk/ICT_DSA_NOTES/blob/main/inter_assessment_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split
from lightgbm import LGBMClassifier
from sklearn.metrics import f1_score, classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import HistGradientBoostingClassifier

In [ ]:
df=pd.read_csv('/content/train_LZdllcl.csv')
df.head()

,employee_id,department,region,education,gender,recruitment_channel,no_of_trainings,age,previous_year_rating,length_of_service,KPIs_met >80%,awards_won?,avg_training_score,is_promoted
0,65438,Sales & Marketing,region_7,Master's & above,f,sourcing,1,35,5.0,8,1,0,49,0
1,65141,Operations,region_22,Bachelor's,m,other,1,30,5.0,4,0,0,60,0
2,7513,Sales & Marketing,region_19,Bachelor's,m,sourcing,1,34,3.0,7,0,0,50,0
3,2542,Sales & Marketing,region_23,Bachelor's,m,other,2,39,1.0,10,0,0,50,0
4,48945,Technology,region_26,Bachelor's,m,other,1,45,3.0,2,0,0,73,0


In [ ]:
df.isnull().sum()

,0
employee_id,0
department,0
region,0
education,2409
gender,0
recruitment_channel,0
no_of_trainings,0
age,0
previous_year_rating,4124
length_of_service,0


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54808 entries, 0 to 54807
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   employee_id           54808 non-null  int64  
 1   department            54808 non-null  object 
 2   region                54808 non-null  object 
 3   education             52399 non-null  object 
 4   gender                54808 non-null  object 
 5   recruitment_channel   54808 non-null  object 
 6   no_of_trainings       54808 non-null  int64  
 7   age                   54808 non-null  int64  
 8   previous_year_rating  50684 non-null  float64
 9   length_of_service     54808 non-null  int64  
 10  KPIs_met >80%         54808 non-null  int64  
 11  awards_won?           54808 non-null  int64  
 12  avg_training_score    54808 non-null  int64  
 13  is_promoted           54808 non-null  int64  
dtypes: float64(1), int64(8), object(5)
memory usage: 5.9+ MB


In [ ]:
df.describe()

,employee_id,no_of_trainings,age,previous_year_rating,length_of_service,KPIs_met >80%,awards_won?,avg_training_score,is_promoted
count,54808.000000,54808.000000,54808.000000,50684.000000,54808.000000,54808.000000,54808.000000,54808.000000,54808.000000
mean,39195.830627,1.253011,34.803915,3.329256,5.865512,0.351974,0.023172,63.386750,0.085170
std,22586.581449,0.609264,7.660169,1.259993,4.265094,0.477590,0.150450,13.371559,0.279137
min,1.000000,1.000000,20.000000,1.000000,1.000000,0.000000,0.000000,39.000000,0.000000
25%,19669.750000,1.000000,29.000000,3.000000,3.000000,0.000000,0.000000,51.000000,0.000000
50%,39225.500000,1.000000,33.000000,3.000000,5.000000,0.000000,0.000000,60.000000,0.000000
75%,58730.500000,1.000000,39.000000,4.000000,7.000000,1.000000,0.000000,76.000000,0.000000
max,78298.000000,10.000000,60.000000,5.000000,37.000000,1.000000,1.000000,99.000000,1.000000


In [ ]:
df.duplicated().sum()

np.int64(0)

In [ ]:
df.shape

(54808, 14)

In [ ]:
#preprocessing
train_df=df.copy()

train_df['education'] =train_df['education'].fillna(train_df['education'].mode()[0])
train_df['previous_year_rating'].fillna(train_df['previous_year_rating'].median(),inplace=True)


/tmp/ipykernel_1220/3590708258.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train_df['previous_year_rating'].fillna(train_df['previous_year_rating'].median(),inplace=True)


In [ ]:
cat_cols=['department','region','education','gender','recruitment_channel']
df_encoded=pd.get_dummies(train_df,columns=cat_cols,drop_first=True,dtype=int)

In [ ]:
X=df_encoded.drop(columns=['employee_id','is_promoted'])
y=df_encoded['is_promoted']

In [ ]:
#lgbm model
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    scale_pos_weight=2.5,
    random_state=42
)

model.fit(X_train, y_train)
val_probs = model.predict_proba(X_val)[:, 1]
best_thresh = 0.5
best_f1 = 0.0

for thresh in np.arange(0.2, 0.6, 0.02):
    preds = (val_probs >= thresh).astype(int)
    score = f1_score(y_val, preds)
    if score > best_f1:
        best_f1 = score
        best_thresh = thresh

print(f"Optimal Decision Threshold: {best_thresh:.2f}")
print(f"Best Validation F1-Score: {best_f1:.4f}\n")
val_preds = (val_probs >= best_thresh).astype(int)
print(classification_report(y_val, val_preds))

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 3734, number of negative: 40112
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005827 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 251
[LightGBM] [Info] Number of data points in the train set: 43846, number of used features: 53
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.085162 -> initscore=-2.374195
[LightGBM] [Info] Start training from score -2.374195
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best 

In [ ]:
#random Forest Classifier
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)
val_probs_rf = rf_model.predict_proba(X_val)[:, 1]
best_thresh_rf = 0.5
best_f1_rf = 0.0

for thresh in np.arange(0.2, 0.6, 0.02):
    preds = (val_probs_rf >= thresh).astype(int)
    score = f1_score(y_val, preds)
    if score > best_f1_rf:
        best_f1_rf = score
        best_thresh_rf = thresh

print(f"--- Random Forest Results ---")
print(f"Optimal Threshold: {best_thresh_rf:.2f}")
print(f"Best Validation F1-Score: {best_f1_rf:.4f}\n")

val_preds_rf = (val_probs_rf >= best_thresh_rf).astype(int)
print(classification_report(y_val, val_preds_rf))

--- Random Forest Results ---
Optimal Threshold: 0.58
Best Validation F1-Score: 0.3788

              precision    recall  f1-score   support

           0       0.96      0.84      0.90     10028
           1       0.27      0.64      0.38       934

    accuracy                           0.82     10962
   macro avg       0.62      0.74      0.64     10962
weighted avg       0.90      0.82      0.85     10962



In [ ]:
#histgb model
hgb_model = HistGradientBoostingClassifier(
    max_iter=300,
    learning_rate=0.05,
    class_weight='balanced',
    random_state=42
)

hgb_model.fit(X_train, y_train)

val_probs_hgb = hgb_model.predict_proba(X_val)[:, 1]
best_thresh_hgb = 0.5
best_f1_hgb = 0.0

for thresh in np.arange(0.2, 0.6, 0.02):
    preds = (val_probs_hgb >= thresh).astype(int)
    score = f1_score(y_val, preds)
    if score > best_f1_hgb:
        best_f1_hgb = score
        best_thresh_hgb = thresh

print(f"--- HistGradientBoosting Results ---")
print(f"Optimal Threshold: {best_thresh_hgb:.2f}")
print(f"Best Validation F1-Score: {best_f1_hgb:.4f}\n")
val_preds_hgb = (val_probs_hgb >= best_thresh_hgb).astype(int)
print(classification_report(y_val, val_preds_hgb))

--- HistGradientBoosting Results ---
Optimal Threshold: 0.58
Best Validation F1-Score: 0.3995

              precision    recall  f1-score   support

           0       0.98      0.78      0.87     10028
           1       0.26      0.84      0.40       934

    accuracy                           0.78     10962
   macro avg       0.62      0.81      0.63     10962
weighted avg       0.92      0.78      0.83     10962



In [ ]:
param_grid = {
    'n_estimators': [200, 300],
    'learning_rate': [0.05, 0.1],
    'max_depth': [6, 8],
    'num_leaves': [15, 31],
    'scale_pos_weight': [2.5, 3.0],
    'subsample': [0.8, 0.9],
    'colsample_bytree': [0.8]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid_search = GridSearchCV(
    estimator=LGBMClassifier(random_state=42, verbose=-1),
    param_grid=param_grid,
    scoring='f1',
    cv=cv,
    n_jobs=-1
)

print("Starting GridSearchCV on LightGBM...")
grid_search.fit(X_train, y_train)

print("\nBest Hyperparameters from GridSearchCV:")
print(grid_search.best_params_)
best_lgb = grid_search.best_estimator_
val_probs_grid = best_lgb.predict_proba(X_val)[:, 1]

best_thresh_grid = 0.5
best_f1_grid = 0.0

for thresh in np.arange(0.20, 0.65, 0.02):
    preds = (val_probs_grid >= thresh).astype(int)
    score = f1_score(y_val, preds)
    if score > best_f1_grid:
        best_f1_grid = score
        best_thresh_grid = thresh

print(f"\n--- GridSearchCV LightGBM Results ---")
print(f"Optimal Threshold: {best_thresh_grid:.2f}")
print(f"Best Validation F1-Score: {best_f1_grid:.4f}\n")

val_preds_grid = (val_probs_grid >= best_thresh_grid).astype(int)
print(classification_report(y_val, val_preds_grid))

Starting GridSearchCV on LightGBM...

Best Hyperparameters from GridSearchCV:
{'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 8, 'n_estimators': 200, 'num_leaves': 15, 'scale_pos_weight': 3.0, 'subsample': 0.8}

--- GridSearchCV LightGBM Results ---
Optimal Threshold: 0.54
Best Validation F1-Score: 0.5318

              precision    recall  f1-score   support

           0       0.95      0.99      0.97     10028
           1       0.76      0.41      0.53       934

    accuracy                           0.94     10962
   macro avg       0.86      0.70      0.75     10962
weighted avg       0.93      0.94      0.93     10962



test data using

In [ ]:
test_df=pd.read_csv('/content/test_2umaH9m.csv')
sample_sub= pd.read_csv('/content/sample_submission_M0L0uXE.csv')

test_clean=test_df.copy()
test_clean['education'] =test_clean['education'].fillna(test_clean['education'].mode()[0])
test_clean['previous_year_rating'].fillna(test_clean['previous_year_rating'].median(),inplace=True)

/tmp/ipykernel_1220/511573210.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  test_clean['previous_year_rating'].fillna(test_clean['previous_year_rating'].median(),inplace=True)


In [ ]:
cat_cols = ['department', 'region', 'education', 'gender', 'recruitment_channel']
test_encoded = pd.get_dummies(test_clean, columns=cat_cols, drop_first=True, dtype=int)

In [ ]:
X_test = test_encoded.drop(columns=['employee_id'], errors='ignore').reindex(columns=X.columns, fill_value=0)
test_probs = best_lgb.predict_proba(X_test)[:, 1]
test_preds = (test_probs >= 0.54).astype(int)

submission = sample_sub.copy()
submission['is_promoted'] = test_preds
submission.to_csv('final_submission.csv', index=False)

print("'final_submission.csv' generated successfully!")
print(f"Total test rows processed: {len(submission)}")
print("\nPredicted Promotion Class Breakdown:")
print(submission['is_promoted'].value_counts())

'final_submission.csv' generated successfully!
Total test rows processed: 23490

Predicted Promotion Class Breakdown:
is_promoted
0    22460
1     1030
Name: count, dtype: int64
